In [ ]:
import pandas as pd
import numpy as np
import colorsys
import matplotlib.pyplot as plt

def analyze_genres_colors(df):
    # 1. Příprava: Rozbalíme žánry (protože hra jich má víc)
    df_genres = df.copy()
    df_genres['Genres'] = df_genres['Genres'].fillna('').str.split('|')
    df_exploded = df_genres.explode('Genres')
    # Odstraníme prázdné žánry
    df_exploded = df_exploded[df_exploded['Genres'] != ""]

    def get_hsv_metrics(row):
        sats = []
        lums = []
        for i in range(1, 6):
            r, g, b = row[f'C{i}_R'], row[f'C{i}_G'], row[f'C{i}_B']
            if pd.notnull(r):
                # Převod na 0-1 pro colorsys
                r_n, g_n, b_n = r/255.0, g/255.0, b/255.0
                # HSV: Hue (odstín), Saturation (sytost), Value (jas)
                h, s, v = colorsys.rgb_to_hsv(r_n, g_n, b_n)
                # Luminance (vnímání jasu okem)
                lum = 0.299*r + 0.587*g + 0.114*b
                
                sats.append(s)
                lums.append(lum)
        
        return pd.Series([np.mean(sats), np.mean(lums)])

    # Aplikujeme výpočet na barevné sloupce
    metrics = df_exploded.apply(get_hsv_metrics, axis=1)
    df_exploded[['Avg_Saturation', 'Avg_Luminance']] = metrics

    return df_exploded

# Načtení dat a analýza
df = pd.read_csv('data/game_data.csv')
analyzed_df = analyze_genres_colors(df)

# Seskupení podle dekády a žánru pro porovnání
summary = analyzed_df.groupby(['Decade', 'Genres'])[['Avg_Saturation', 'Avg_Luminance']].mean().reset_index()
print(summary)


In [ ]:
plt.figure(figsize=(12, 7))

# Filtrujeme roky, které jste zpracovala
for decade in [1980, 2000]:
    data = summary[summary['Decade'] == decade]
    plt.scatter(data['Avg_Luminance'], data['Avg_Saturation'], 
                label=f'Dekáda {decade}s', s=100, alpha=0.7)
    
    # Přidání popisků žánrů k bodům
    for i, row in data.iterrows():
        plt.annotate(row['Genres'], (row['Avg_Luminance'], row['Avg_Saturation']), 
                     textcoords="offset points", xytext=(0,10), ha='center', fontsize=9)

plt.title('Změna barevnosti žánrů: 1980s vs 2000s', fontsize=15)
plt.xlabel('Průměrný Jas (Luminance) -> vyšší je světlejší', fontsize=12)
plt.ylabel('Průměrná Sytost (Saturation) -> vyšší je barevnější', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()
